# 公開データの再分析 1: 関節角度の誤差の偏りとばらつき

OpenCap の関節角度の誤差を公開妥当化データで再分析するノートブックです。

## 目的

OpenCap の関節角度を力学推定の入力として用いるとき、その誤差が力学推定にどの程度影響しうるかを見積もります。
誤差は、力学推定への伝わり方が異なる 2 つの成分に分けます。

- **試行内で一定の偏り（バイアス）**: 時間微分で消え、姿勢の違いとしてモーメントアームや筋長に伝わる。
- **試行内のばらつき（標準偏差）**: 数値微分で増幅され、速度・加速度の追跡項を通じて最適化に直接影響する。

## データ

Uhlrich et al. (2023, PLOS Computational Biology, doi:10.1371/journal.pcbi.1011462) が公開する OpenCap の実験室妥当化データセットを用います。同論文の Data Availability に従い、https://simtk.org/projects/opencap から `LabValidation_withoutVideos` を取得し、環境変数 `LABVALIDATION_DIR` でその場所を指定してください（既定は `public_data/data/LabValidation_withoutVideos`）。
データが手元にない場合は、同梱の `results/*.csv`（著者の実行結果）を読み込んで表を再構成します。

## 方法

- 被験者 10 名（subject2〜subject11）、2 カメラ・HRNet 構成。試行は被験者あたり 歩行 3、スクワット 1、立ち座り 1、ドロップジャンプ 3 の計 80 試行。
- 対象は実行例の最適化が追跡する左右の股関節屈曲・膝屈曲・足関節背屈の 6 つの関節角度。
- 参照値は光学式モーションキャプチャの逆運動学、推定値は OpenCap の逆運動学（いずれも 100 Hz、公開された出力のまま）。両系列の時刻を 1 ms に丸めて同じ時刻のフレームを対応づけ、一方の系列にしかないフレームは除外する。
- 被験者、試行、関節角度の組み合わせを 1 組とする 480 組について、フレームごとの差 $d_j$ = 推定値 − 参照値 から、バイアス = mean($d$)、標準偏差 = SD($d$)（$n$ で割る定義）、RMSE = $\sqrt{\mathrm{mean}(d^2)}$ を求める。この定義では RMSE² = バイアス² + 標準偏差² が恒等的に成り立つ。
- 集約は (a) 480 組の指標を組ごとに等しく重み付けして平均と組間の標準偏差を求める方法と、(b) 全組のフレームをまとめて各指標を求める方法の 2 通り。(b) ではフレーム数の多い試行の重みが大きく、標準偏差には組ごとの偏りの違いも含まれる。


In [1]:
import os, sys, platform, numpy as np, pandas as pd
HERE = os.path.abspath('.'); sys.path.insert(0, os.path.join(HERE, 'scripts'))
DATA_DIR = os.environ.get('LABVALIDATION_DIR', os.path.join(HERE, 'data', 'LabValidation_withoutVideos'))
HAVE_DATA = os.path.isdir(os.path.join(DATA_DIR, 'subject2'))
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__)
print('データの場所:', DATA_DIR, '| 利用可能:', HAVE_DATA)

Python 3.11.17 | numpy 2.4.6 | pandas 3.0.6
データの場所: /data/LabValidation_withoutVideos | 利用可能: True


## 1. 解析の実行（データがある場合）

`scripts/joint_angles.py` の `analyze()` を呼び、組ごとの表 `U`（480 行）、全フレームの差 `F`、集約表 `S` を得て `results/` に保存します。データがない場合は同梱の結果を読み込みます。

In [2]:
if HAVE_DATA:
    from joint_angles import analyze
    U, F, S = analyze(DATA_DIR)
    os.makedirs('results', exist_ok=True)
    U.to_csv('results/joint_angles_units.csv', index=False); S.to_csv('results/joint_angles_summary.csv', index=False)
    print('被験者 %d, 試行 %d, 組 %d' % (U.subject.nunique(), U.groupby(['subject', 'trial']).ngroups, len(U)))
else:
    U = pd.read_csv('results/joint_angles_units.csv'); S = pd.read_csv('results/joint_angles_summary.csv')
    print('同梱の結果を読み込みました: 組 %d' % len(U))

被験者 10, 試行 80, 組 480


## 2. 結果表の再構成

「組ごとの値の平均」の 3 列は RMSE・バイアス・標準偏差の組ごとの値の平均（括弧内は組間の標準偏差）、「全フレーム」の 3 列は全組のフレームをまとめた RMSE・バイアス・標準偏差です。単位は度。

In [3]:
LAB = {'all': '全体', 'walking': '歩行', 'squats': 'スクワット', 'STS': '立ち座り', 'DJ': 'ドロップジャンプ',
       'hip_flexion_r': '股関節屈曲（右）', 'hip_flexion_l': '股関節屈曲（左）', 'knee_angle_r': '膝屈曲（右）',
       'knee_angle_l': '膝屈曲（左）', 'ankle_angle_r': '足関節背屈（右）', 'ankle_angle_l': '足関節背屈（左）'}
ORDER = list(LAB)
T = S.set_index('group').loc[ORDER]
tab6 = pd.DataFrame({
    '区分': [LAB[g] for g in T.index], '組数': T.n_units.values,
    '組ごと RMSE (SD)': ['%.2f (%.2f)' % (a, b) for a, b in zip(T.rmse_unit_mean, T.rmse_unit_sd)],
    '組ごと バイアス (SD)': ['%+.2f (%.2f)' % (a, b) for a, b in zip(T.bias_unit_mean, T.bias_unit_sd)],
    '組ごと 標準偏差 (SD)': ['%.2f (%.2f)' % (a, b) for a, b in zip(T.sd_unit_mean, T.sd_unit_sd)],
    '全フレーム RMSE': T.rmse_pooled.round(2).values, '全フレーム バイアス': ['%+.2f' % x for x in T.bias_pooled],
    '全フレーム 標準偏差': T.sd_pooled.round(2).values}).set_index('区分')
pd.set_option('display.width', 200); display(tab6)

,組数,組ごと RMSE (SD),組ごと バイアス (SD),組ごと 標準偏差 (SD),全フレーム RMSE,全フレーム バイアス,全フレーム 標準偏差
区分,,,,,,,
全体,480,6.15 (2.98),+2.38 (4.49),4.15 (1.92),5.83,+1.38,5.66
歩行,180,5.39 (2.50),+2.82 (3.97),3.21 (1.15),5.95,+2.72,5.29
スクワット,60,4.48 (2.09),+0.76 (3.43),3.21 (1.42),4.83,+0.75,4.78
立ち座り,60,5.35 (2.55),+1.21 (4.58),3.36 (1.31),5.99,+1.25,5.86
ドロップジャンプ,180,7.73 (3.12),+2.86 (5.08),5.66 (1.90),8.31,+2.88,7.80
股関節屈曲（右）,80,6.82 (2.91),+3.84 (5.04),3.65 (1.32),7.11,+3.74,6.05
股関節屈曲（左）,80,7.08 (2.91),+3.75 (5.06),4.09 (1.55),7.62,+3.74,6.64
膝屈曲（右）,80,4.66 (1.63),-1.03 (2.89),3.60 (1.44),4.37,-2.21,3.77
膝屈曲（左）,80,4.66 (1.76),-0.48 (2.47),3.94 (1.76),4.19,-1.31,3.98


## 3. 誤差の 2 成分の大きさ

組ごとのバイアスの絶対値の平均と、試行内のばらつきを表す標準偏差の平均（度）。

In [4]:
A = S.set_index('group').loc[ORDER[:5]]
display(pd.DataFrame({'RMSE': A.rmse_unit_mean, '|バイアス|': A.absbias_unit_mean, '標準偏差': A.sd_unit_mean}).rename(index=LAB).round(2))

,RMSE,|バイアス|,標準偏差
group,,,
全体,6.15,4.00,4.15
歩行,5.39,3.90,3.21
スクワット,4.48,2.75,3.21
立ち座り,5.35,3.69,3.36
ドロップジャンプ,7.73,4.61,5.66


## 備考

- `results/joint_angles_units.csv` は 480 組それぞれの n、bias、sd、rmse、mae、`results/joint_angles_summary.csv` は上表の元データです。
- コマンドラインから実行する場合: `LABVALIDATION_DIR=/path/to/LabValidation_withoutVideos python3 scripts/joint_angles.py results`